In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
import lightgbm as lgb

In [19]:
df = pd.read_csv("../data/raw/application_train.csv")
print(df.shape)

(307511, 122)


In [20]:
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

print(df["DAYS_EMPLOYED"].max())
print(df.groupby("DAYS_EMPLOYED_ANOM")["TARGET"].mean())

0.0
DAYS_EMPLOYED_ANOM
0    0.086600
1    0.053996
Name: TARGET, dtype: float64


C:\Users\jayra\AppData\Local\Temp\ipykernel_15224\1532282885.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)


In [21]:
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])

cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
print(len(cat_cols), "text columns")

for c in cat_cols:
    X[c] = X[c].astype("category")

16 text columns


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(246008, 121) (61503, 121)
0.08072908198107379 0.08072776937710356


In [23]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

print("Accuracy: ", accuracy_score(y_test, dummy.predict(X_test)))
print("ROC-AUC: ", roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1])) # type: ignore

Accuracy:  0.9192722306228964
ROC-AUC:  0.5


In [24]:
results = pd.DataFrame({"y": y_test.values, "risk": probs})
results = results.sort_values("risk", ascending=False).reset_index(drop=True)

total_defaulters = results["y"].sum()
print("Defaulters in test set:", total_defaulters)

for pct in [5, 10, 20, 30]:
    n = int(len(results) * pct / 100)
    caught = results.head(n)["y"].sum()
    print(f"Reject riskiest {pct:>2}% -> catches {caught / total_defaulters:.1%} of all defaulters")

Defaulters in test set: 4965
Reject riskiest  5% -> catches 21.1% of all defaulters
Reject riskiest 10% -> catches 34.5% of all defaulters
Reject riskiest 20% -> catches 52.6% of all defaulters
Reject riskiest 30% -> catches 65.0% of all defaulters


In [25]:
model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose= -1)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1] # type: ignore
print("ROC-AUC: ", roc_auc_score(y_test, probs))
print("PR-AUC: ", average_precision_score(y_test, probs))

ROC-AUC:  0.7596430451983794
PR-AUC:  0.25073411329881634


In [26]:
imp = pd.Series(model.feature_importances_, index= X.columns).sort_values(ascending=False)
print(imp.head(15))

ORGANIZATION_TYPE         1102
EXT_SOURCE_1               572
EXT_SOURCE_3               564
EXT_SOURCE_2               491
DAYS_EMPLOYED              381
AMT_CREDIT                 365
DAYS_BIRTH                 348
AMT_ANNUITY                325
DAYS_ID_PUBLISH            322
DAYS_REGISTRATION          286
AMT_GOODS_PRICE            275
DAYS_LAST_PHONE_CHANGE     273
OWN_CAR_AGE                198
OCCUPATION_TYPE            189
AMT_INCOME_TOTAL           187
dtype: int32
